``AAPred().predict_proba()`` scores a **precomputed feature matrix** ``X`` with the fitted deployment ensemble. It is the matrix-level counterpart of :meth:`predict`: where ``predict`` takes raw sequences (``df_seq``) and rebuilds the feature matrix internally, ``predict_proba`` scores an ``X`` you already hold, so a matrix built once (e.g. a large candidate set such as a substratome) is scored directly without re-featurizing and with no ``df_feat`` bound. Where :meth:`predict_oof` scores the *training* set out-of-fold, ``predict_proba`` scores *new* samples with models fit on all data (see [Breimann25]_). We first build the ``DOM_GSEC`` feature matrix and hold out a few proteins as new candidates:

In [1]:
import aaanalysis as aa
aa.options["verbose"] = False  # Disable verbosity

# DOM_GSEC example dataset + its feature set (see [Breimann25]_)
df_seq = aa.load_dataset(name="DOM_GSEC")
labels = df_seq["label"].to_list()
df_feat = aa.load_features(name="DOM_GSEC").head(20)

# Build the CPP feature matrix once, then split off a few proteins as new candidates
sf = aa.SequenceFeature()
X = sf.feature_matrix(features=df_feat["feature"], df_parts=sf.get_df_parts(df_seq=df_seq))
X_train, y_train = X[:-10], labels[:-10]
X_new = X[-10:]

Fit the deployment ensemble on the training matrix, then score the precomputed candidate matrix ``X`` directly. ``predict_proba`` needs a prior :meth:`fit` but **no** bound ``df_feat`` (it works purely on the matrix), and returns one row per sample (row-aligned with ``X``) with a positive-class ``score`` averaged over the ensemble and its ``score_std`` across models:

In [2]:
aap = aa.AAPred(models=["rf", "extra_trees", "log_reg"], random_state=42).fit(X_train, y_train)
df_pred = aap.predict_proba(X=X_new)
aa.display_df(df_pred, n_rows=10, show_shape=True)

DataFrame shape: (10, 2)


,score,score_std
1,0.178376,0.019353
2,0.121250,0.052709
3,0.573780,0.033094
4,0.039724,0.021478
5,0.615094,0.025532
6,0.079682,0.048601
7,0.909509,0.008194
8,0.800845,0.015338
9,0.323732,0.062788
10,0.045826,0.020446


``score_range='percent'`` returns the same scores on a ``[0, 100]`` scale instead of the native ``[0, 1]`` probabilities:

In [3]:
df_pred = aap.predict_proba(X=X_new, score_range="percent")
aa.display_df(df_pred, n_rows=10, show_shape=True)

DataFrame shape: (10, 2)


,score,score_std
1,17.837637,1.935255
2,12.125050,5.270860
3,57.377965,3.309438
4,3.972406,2.147790
5,61.509402,2.553229
6,7.968226,4.860117
7,90.950852,0.819450
8,80.084526,1.533751
9,32.373214,6.278838
10,4.582593,2.044581


This collapses the manual ensemble scoring, ``np.vstack([m.predict_proba(X_new)[:, 1] for m in models]).mean(0)``, into one call and reports the model-agreement ``score_std`` alongside. A single model simply returns a ``score_std`` of ``0``:

In [4]:
df_pred = aa.AAPred(models="rf", random_state=42).fit(X_train, y_train).predict_proba(X=X_new)
aa.display_df(df_pred, n_rows=10, show_shape=True)

DataFrame shape: (10, 2)


,score,score_std
1,0.160000,0.000000
2,0.070000,0.000000
3,0.610000,0.000000
4,0.010000,0.000000
5,0.640000,0.000000
6,0.020000,0.000000
7,0.900000,0.000000
8,0.790000,0.000000
9,0.340000,0.000000
10,0.070000,0.000000
